In [ ]:
import sys
sys.path[:0] = [
    "/workspace/unsupervised-truth-probes",
    "/workspace/unsupervised-truth-probes/src",
]

import os, json, random
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

# Optional LoRA
try:
    from peft import LoraConfig, get_peft_model, TaskType
    PEFT_AVAILABLE = True
except Exception:
    PEFT_AVAILABLE = False

# Prefer local loader mapping model keys to cached paths, if available
try:
    from utils import load_model as _load_model
except Exception:
    _load_model = None

from config import prompt_formats as PROMPT_FORMATS


In [ ]:
# --- Configuration ---
MODEL_NAME = "llama-3-1-70b"  # key for local cache via utils.load_model, or HF id
TRAIN_PATH = "/workspace/unsupervised-truth-probes/data/train_alpaca.json"
EVAL_PATH  = "/workspace/unsupervised-truth-probes/data/test_alpaca.json"
OUTPUT_DIR = "/workspace/unsupervised-truth-probes/outputs/sft-alpaca"

USE_LORA   = True   # set False if peft is unavailable
LORA_R     = 128
LORA_ALPHA = 64
LORA_DROPOUT = 0.0

LEARNING_RATE = 5e-5
NUM_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE  = 4
GRADIENT_ACCUMULATION_STEPS = 4
MAX_LENGTH = 1024
BF16 = True  # use bf16 if supported, else Trainer will use fp16 fallback below
LOGGING_STEPS = 20
EVAL_STRATEGY = "epoch"
SAVE_STRATEGY = "epoch"

# Subsample sizes
TRAIN_SIZE = 2048
EVAL_SIZE = 2048

# Use pairwise prompt format from config
PAIRWISE_FORMAT = PROMPT_FORMATS["PAIRWISE_FORMAT"]
PROMPT_FORMAT = PAIRWISE_FORMAT

os.makedirs(OUTPUT_DIR, exist_ok=True)
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# --- Helpers: parsing, dataset, collator, metrics, LoRA wrap, model loader ---
from typing import Tuple

def _parse_truth_label(label_val: Any) -> int:
    if isinstance(label_val, bool):
        return 1 if label_val else 0
    if isinstance(label_val, (int, float)):
        return 1 if int(label_val) != 0 else 0
    if isinstance(label_val, str):
        lv = label_val.strip().lower()
        if lv in {"true", "t", "1", "yes"}:
            return 1
        if lv in {"false", "f", "0", "no"}:
            return 0
    return 0


def build_alpaca_pair_sft_examples(data: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    examples: List[Dict[str, str]] = []
    for ex in data:
        q = ex.get("question", "")
        c = ex.get("choice", "")
        c2 = ex.get("choice_2", ex.get("choice2", ""))
        label_int = _parse_truth_label(ex.get("label"))
        prompt = PROMPT_FORMAT.format(question=q, choice=c, choice_2=c2)
        answer = " True" if label_int == 1 else " False"
        examples.append({"prompt": prompt, "answer": answer})
    return examples


class AlpacaPairSFTDataset(Dataset):
    def __init__(self, tokenizer: AutoTokenizer, examples: List[Dict[str, str]], max_length: int = 1024):
        self.tokenizer = tokenizer
        self.max_length = int(max_length)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        samples: List[Dict[str, Any]] = []
        for ex in examples:
            prompt = ex["prompt"]
            answer = ex["answer"]
            prompt_ids = self.tokenizer(prompt, add_special_tokens=False).input_ids
            answer_ids = self.tokenizer(answer, add_special_tokens=False).input_ids
            total_len = len(prompt_ids) + len(answer_ids)
            if total_len > self.max_length:
                keep_prompt = max(0, self.max_length - len(answer_ids))
                prompt_ids = prompt_ids[-keep_prompt:] if keep_prompt > 0 else []
            input_ids = prompt_ids + answer_ids
            attention_mask = [1] * len(input_ids)
            labels = ([-100] * len(prompt_ids)) + list(answer_ids)
            samples.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
            })
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]


class DataCollatorForSFT:
    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        pad_id = self.tokenizer.pad_token_id
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids, attention_mask, labels = [], [], []
        for f in features:
            l = len(f["input_ids"])
            pad_len = max_len - l
            input_ids.append(f["input_ids"] + ([pad_id] * pad_len))
            attention_mask.append(f["attention_mask"] + ([0] * pad_len))
            labels.append(f["labels"] + ([-100] * pad_len))
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def compute_first_answer_token_accuracy(eval_preds) -> Dict[str, float]:
    logits, labels = eval_preds
    if isinstance(logits, (list, tuple)):
        logits = logits[0]
    logits = torch.tensor(logits)
    labels = torch.tensor(labels)
    B, T, V = logits.shape
    correct = 0
    total = 0
    with torch.no_grad():
        for i in range(B):
            label_row = labels[i]
            pos = (label_row != -100).nonzero(as_tuple=False)
            if pos.numel() == 0:
                continue
            j = int(pos[0].item())
            pred_token = int(torch.argmax(logits[i, j]))
            if pred_token == int(label_row[j].item()):
                correct += 1
            total += 1
    acc = correct / total if total > 0 else 0.0
    return {"first_answer_token_accuracy": float(acc)}


def maybe_wrap_lora(model: AutoModelForCausalLM) -> AutoModelForCausalLM:
    if not USE_LORA:
        return model
    if not PEFT_AVAILABLE:
        print("[warn] peft not available; proceeding without LoRA")
        return model
    cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "v_proj"],
    )
    return get_peft_model(model, cfg)


def load_model_and_tokenizer(model_name: str) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    if _load_model is not None:
        model, tokenizer = _load_model(model_name)
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name, trust_remote_code=True, device_map="auto" if torch.cuda.is_available() else None
        )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    return model, tokenizer


In [ ]:
# --- Model & tokenizer ---
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
model = maybe_wrap_lora(model)
print(f"Loaded model: hidden={model.config.hidden_size} layers={model.config.num_hidden_layers}")
print(f"Using LoRA: {USE_LORA and PEFT_AVAILABLE}")


In [ ]:
# --- Load data & build examples ---
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_raw = json.load(f)
with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_raw = json.load(f)

# Downsample to requested sizes (deterministic via SEED)
if isinstance(train_raw, list) and len(train_raw) > int(TRAIN_SIZE):
    train_raw = random.sample(train_raw, int(TRAIN_SIZE))
if isinstance(eval_raw, list) and len(eval_raw) > int(EVAL_SIZE):
    eval_raw = random.sample(eval_raw, int(EVAL_SIZE))

train_examples = build_alpaca_pair_sft_examples(train_raw)
eval_examples  = build_alpaca_pair_sft_examples(eval_raw)

print(f"Train examples: {len(train_examples)} | Eval examples: {len(eval_examples)}")
print("Example pairwise prompt:")
print(train_examples[0]["prompt"]) 
print("Example answer:", train_examples[0]["answer"])


In [ ]:
# --- Manual finetuning loop (no Trainer) ---
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader

# IDs for " True" / " False" tokens
true_id  = tokenizer(" True",  add_special_tokens=False).input_ids[0]
false_id = tokenizer(" False", add_special_tokens=False).input_ids[0]

def format_target(example: Dict[str, Any]) -> str:
    return PROMPT_FORMAT.format(question=example["question"], choice=example["choice"], choice_2=example.get("choice_2", ""))

@torch.no_grad()
def score_margins(prompts: List[str], batch_size: int = 32) -> List[float]:
    margins: List[float] = []
    device = next(model.parameters()).device
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(device)
        logits = model(**enc).logits
        lens   = enc["attention_mask"].sum(dim=1)
        last   = logits[torch.arange(len(batch)), lens-1]
        logp   = torch.log_softmax(last, dim=-1)
        margins.extend((logp[:, true_id] - logp[:, false_id]).detach().cpu().tolist())
    return margins

# Build supervised train/eval targets
train_sup = [{
    "question": ex.get("question", ""),
    "choice": ex.get("choice", ""),
    "choice_2": ex.get("choice_2", ex.get("choice2", "")),
    "label": 1 if _parse_truth_label(ex.get("label")) == 1 else 0,
} for ex in train_raw]

eval_sup = [{
    "question": ex.get("question", ""),
    "choice": ex.get("choice", ""),
    "choice_2": ex.get("choice_2", ex.get("choice2", "")),
    "label": 1 if _parse_truth_label(ex.get("label")) == 1 else 0,
} for ex in eval_raw]

train_prompts = [format_target(ex) for ex in train_sup]
train_labels  = np.array([ex["label"] for ex in train_sup], dtype=int)

eval_prompts = [format_target(ex) for ex in eval_sup]
eval_labels  = np.array([ex["label"] for ex in eval_sup], dtype=int)

# DataLoader over indices for shuffling
indices = list(range(len(train_prompts)))
train_loader = DataLoader(indices, batch_size=PER_DEVICE_TRAIN_BATCH_SIZE, shuffle=True)

device = next(model.parameters()).device
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

global_step = 0
for epoch in range(int(NUM_EPOCHS)):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch_idx in enumerate(train_loader):
        batch_idx = batch_idx.tolist() if hasattr(batch_idx, 'tolist') else list(batch_idx)
        batch_prompts = [train_prompts[i] for i in batch_idx]
        gold = torch.tensor(train_labels[batch_idx], dtype=torch.long, device=device)

        enc = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(device)

        logits = model(**enc).logits
        lens   = enc["attention_mask"].sum(dim=1)
        last   = logits[torch.arange(len(batch_prompts), device=device), lens-1]
        pair_logits = torch.stack([last[:, false_id], last[:, true_id]], dim=1)
        loss = F.cross_entropy(pair_logits, gold)

        loss = loss / max(1, int(GRADIENT_ACCUMULATION_STEPS))
        loss.backward()

        if (step + 1) % max(1, int(GRADIENT_ACCUMULATION_STEPS)) == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        total_loss += float(loss.item())

    avg_loss = total_loss / max(1, len(train_loader))

    # Eval
    model.eval()
    with torch.no_grad():
        margins = np.array(score_margins(eval_prompts, batch_size=PER_DEVICE_EVAL_BATCH_SIZE))
        preds = (margins >= 0).astype(int)
        acc = float((preds == eval_labels).mean())

    print(f"Epoch {epoch+1}: avg_loss={avg_loss:.4f} | Eval Acc@0={acc:.4f} | steps={global_step}")

# Save
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")
